In [2]:
import os
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

BASE = r"C:\Users\padhi\Downloads\siH26052_data"
FEATURE_DIR = os.path.join(BASE, "features")
MODEL_DIR = os.path.join(BASE, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Ready")

Ready


In [3]:
X_train_full = np.load(
    os.path.join(FEATURE_DIR, "X_train.npy"),
    mmap_mode="r"
)

y_train_full = np.load(
    os.path.join(FEATURE_DIR, "y_train.npy"),
    mmap_mode="r"
)

X_dev = np.load(
    os.path.join(FEATURE_DIR, "X_dev.npy"),
    mmap_mode="r"
)

y_dev = np.load(
    os.path.join(FEATURE_DIR, "y_dev.npy"),
    mmap_mode="r"
)

print("X_train:", X_train_full.shape)
print("y_train:", y_train_full.shape)

print("X_dev:", X_dev.shape)
print("y_dev:", y_dev.shape)

X_train: (3217511, 257)
y_train: (3217511, 257)
X_dev: (241529, 257)
y_dev: (241529, 257)


In [5]:
MAX_TRAIN_FRAMES = 500_000

rng = np.random.default_rng(42)

indices = rng.choice(
    X_train_full.shape[0],
    size=MAX_TRAIN_FRAMES,
    replace=False
)

X_train = np.asarray(X_train_full[indices])
y_train = np.asarray(y_train_full[indices])

print("Selected training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

Selected training data:
X_train: (500000, 257)
y_train: (500000, 257)


In [6]:
scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)

X_dev_scaled = scaler_X.transform(X_dev)

print("X scaling complete")

X scaling complete


In [7]:
scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train)

print("y scaling complete")

y scaling complete


In [8]:
model = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    batch_size=256,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
    verbose=True
)

print(model)

MLPRegressor(batch_size=256, early_stopping=True, hidden_layer_sizes=(128, 64),
             max_iter=30, random_state=42, verbose=True)


In [9]:
model.fit(
    X_train_scaled,
    y_train_scaled
)

Iteration 1, loss = 0.14013644
Validation score: 0.797879
Iteration 2, loss = 0.09500292
Validation score: 0.816438
Iteration 3, loss = 0.08837551
Validation score: 0.827154
Iteration 4, loss = 0.08642166
Validation score: 0.815793
Iteration 5, loss = 0.08547133
Validation score: 0.827187
Iteration 6, loss = 0.08479840
Validation score: 0.827122
Iteration 7, loss = 0.08432102
Validation score: 0.831586
Iteration 8, loss = 0.08412558
Validation score: 0.827611
Iteration 9, loss = 0.08380985
Validation score: 0.832731
Iteration 10, loss = 0.08374366
Validation score: 0.832790
Iteration 11, loss = 0.08345457
Validation score: 0.831570
Iteration 12, loss = 0.08339411
Validation score: 0.833865
Iteration 13, loss = 0.08321858
Validation score: 0.833608
Iteration 14, loss = 0.08315054
Validation score: 0.833019
Iteration 15, loss = 0.08289557
Validation score: 0.834855
Iteration 16, loss = 0.08294939
Validation score: 0.832747
Iteration 17, loss = 0.08273895
Validation score: 0.833495
Iterat

C:\Users\padhi\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor(batch_size=256, early_stopping=True, hidden_layer_sizes=(128, 64),
             max_iter=30, random_state=42, verbose=True)

In [11]:
y_dev_pred_scaled = model.predict(X_dev_scaled)

y_dev_pred = scaler_y.inverse_transform(
    y_dev_pred_scaled
)

print("Prediction complete")
print("Prediction shape:", y_dev_pred.shape)

Prediction complete
Prediction shape: (241529, 257)


In [12]:
mse = mean_squared_error(
    y_dev,
    y_dev_pred
)

mae = mean_absolute_error(
    y_dev,
    y_dev_pred
)


print("VALIDATION RESULTS")


print("MSE:", mse)
print("MAE:", mae)

VALIDATION RESULTS
MSE: 0.07805915176868439
MAE: 0.08119124174118042


In [13]:
joblib.dump(
    model,
    os.path.join(MODEL_DIR, "mlp_model.pkl")
)

joblib.dump(
    scaler_X,
    os.path.join(MODEL_DIR, "scaler_X.pkl")
)

joblib.dump(
    scaler_y,
    os.path.join(MODEL_DIR, "scaler_y.pkl")
)

print("Model and scalers saved successfully.")

Model and scalers saved successfully.


In [14]:
import os
import numpy as np
import joblib

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

BASE = r"C:\Users\padhi\Downloads\siH26052_data"

FEATURE_DIR = os.path.join(BASE, "features")
MODEL_DIR = os.path.join(BASE, "model")

X_test = np.load(
    os.path.join(FEATURE_DIR, "X_test.npy"),
    mmap_mode="r"
)

y_test = np.load(
    os.path.join(FEATURE_DIR, "y_test.npy"),
    mmap_mode="r"
)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_test: (247998, 257)
y_test: (247998, 257)


In [15]:
model = joblib.load(
    os.path.join(MODEL_DIR, "mlp_model.pkl")
)

scaler_X = joblib.load(
    os.path.join(MODEL_DIR, "scaler_X.pkl")
)

scaler_y = joblib.load(
    os.path.join(MODEL_DIR, "scaler_y.pkl")
)

print("Model loaded successfully.")

Model loaded successfully.


In [16]:
X_test_scaled = scaler_X.transform(X_test)

print("Test data scaled.")
print(X_test_scaled.shape)

Test data scaled.
(247998, 257)


In [17]:
y_test_pred_scaled = model.predict(
    X_test_scaled
)

print("Prediction complete.")
print(y_test_pred_scaled.shape)

Prediction complete.
(247998, 257)


In [18]:
y_test_pred = scaler_y.inverse_transform(
    y_test_pred_scaled
)

print("Converted predictions back to original scale.")

Converted predictions back to original scale.


In [19]:
test_mse = mean_squared_error(
    y_test,
    y_test_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)


print("TEST RESULTS")


print("MSE :", test_mse)
print("MAE :", test_mae)
print("R²  :", test_r2)

TEST RESULTS
MSE : 0.06369055062532425
MAE : 0.07701955735683441
R²  : 0.8352088332176208


In [21]:
%pip install soundfile

  Using cached soundfile-0.14.0-py2.py3-none-win_amd64.whl.metadata (18 kB)
Using cached soundfile-0.14.0-py2.py3-none-win_amd64.whl (1.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install librosa

  Using cached decorator-5.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
Using cached decorator-5.3.1-py3-none-any.whl (10 kB)
Using cached pooch-1.9.0-py3-none-any.whl (67 kB)

  Attempting uninstall: msgpack

    Found existing installation: msgpack 1.0.3

    Uninstalling msgpack-1.0.3:

      Successfully uninstalled msgpack-1.0.3

   -------- ------------------------------- 1/5 [msgpack]
  Attempting uninstall: decorator
   -------- ------------------------------- 1/5 [msgpack]
    Found existing installation: decorator 5.1.1
   -------- ------------------------------- 1/5 [msgpack]
    Uninstalling decorator-5.1.1:
   -------- ------------------------------- 1/5 [msgpack]
   ---------------- ----------------------- 2/5 [decorator]
   ---------------- ----------------------- 2/5 [decorator]
   ---------------- ----------------------- 2/5 [decorator]
   ---------------- ----------------------- 2/5 [decorator]
   ----------------

In [1]:
import os
import numpy as np
import soundfile as sf
import librosa
import joblib

BASE = r"C:\Users\padhi\Downloads\siH26052_data"

MODEL_DIR = os.path.join(BASE, "model")

model = joblib.load(
    os.path.join(MODEL_DIR, "mlp_model.pkl")
)

scaler_X = joblib.load(
    os.path.join(MODEL_DIR, "scaler_X.pkl")
)

scaler_y = joblib.load(
    os.path.join(MODEL_DIR, "scaler_y.pkl")
)

print("Model loaded.")

Model loaded.


In [2]:
TEST_CLEAN_DIR = os.path.join(
    BASE,
    "test_new",
    "clean"
)

TEST_NOISY_DIR = os.path.join(
    BASE,
    "test_new",
    "noisy"
)

files = [
    f for f in os.listdir(TEST_NOISY_DIR)
    if f.lower().endswith(".wav")
]

files.sort()

filename = files[0]

noisy_path = os.path.join(
    TEST_NOISY_DIR,
    filename
)

clean_path = os.path.join(
    TEST_CLEAN_DIR,
    filename
)

print("Testing file:", filename)

Testing file: 1089-134686-0001.wav


In [3]:
noisy, sr = sf.read(noisy_path)
clean, sr_clean = sf.read(clean_path)

noisy = noisy.astype(np.float32)
clean = clean.astype(np.float32)

print("Sample rate:", sr)
print("Noisy samples:", len(noisy))
print("Clean samples:", len(clean))

Sample rate: 16000
Noisy samples: 52400
Clean samples: 52400


In [4]:
N_FFT = 512
HOP_LENGTH = 256

noisy_stft = librosa.stft(
    noisy,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH
)

noisy_mag = np.abs(noisy_stft)

noisy_phase = np.angle(noisy_stft)

print("STFT shape:", noisy_stft.shape)

STFT shape: (257, 205)


In [5]:
X = noisy_mag.T

print("Input shape:", X.shape)

Input shape: (205, 257)


In [6]:
X_scaled = scaler_X.transform(X)

print("Scaled input shape:", X_scaled.shape)

Scaled input shape: (205, 257)


In [7]:
pred_scaled = model.predict(
    X_scaled
)

print("Prediction shape:", pred_scaled.shape)

Prediction shape: (205, 257)


In [8]:
pred_mag = scaler_y.inverse_transform(
    pred_scaled
)

print("Predicted magnitude shape:", pred_mag.shape)

Predicted magnitude shape: (205, 257)


In [11]:
# Convert prediction from:
# (frames, frequency)
# to:
# (frequency, frames)

pred_mag = pred_mag.T

print("Predicted magnitude:", pred_mag.shape)
print("Noisy phase:", noisy_phase.shape)

# Make sure magnitude is non-negative
pred_mag = np.maximum(pred_mag, 0)

Predicted magnitude: (257, 205)
Noisy phase: (257, 205)


In [12]:
enhanced_stft = (
    pred_mag *
    np.exp(1j * noisy_phase)
)

In [13]:
enhanced_audio = librosa.istft(
    enhanced_stft,
    hop_length=HOP_LENGTH
)

print("Enhanced audio length:", len(enhanced_audio))

Enhanced audio length: 52224


In [15]:
OUTPUT_DIR = os.path.join(
    BASE,
    "enhanced"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [16]:
output_path = os.path.join(
    OUTPUT_DIR,
    "enhanced_test.wav"
)

sf.write(
    output_path,
    enhanced_audio,
    sr
)

print("Saved:")
print(output_path)

Saved:
C:\Users\padhi\Downloads\siH26052_data\enhanced\enhanced_test.wav
